# Rust 102: Logic Blocks and Control Flow

Chapter 02 of the Rust track. Chapter 01 covered how values are named, typed and
printed. This chapter covers how a program **decides** and **repeats**.

The parts are familiar — `if`, `while`, `for`, and something in place of `switch` —
but three things here differ enough from Python and C++ to be worth a whole chapter.
Every construct is an **expression**, so it produces a value. There is **no
truthiness**, so a condition is a `bool` and nothing else. And `match` is not a
prettier `switch`: it is a pattern-matching construct the compiler checks for
completeness.

Everything here is a **runnable example**. Read the markdown, run the cell below it,
and change nothing — the practice twin of this notebook is
`02_logic-blocks-and-control-flow_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, not a beginner. The material is
pitched at someone arriving from **Python** and **C++**, so it spends its words on
where Rust *differs* rather than where it agrees. You already know what a loop is.
What is new is the expression orientation, the absence of truthiness, the loop-label
syntax, `match` as a pattern language, and `if let` / `let ... else` as the tools for
opening up `Option` and `Result`.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach
  it.** Nothing here is convincing until you have seen the output.
- Cells are meant to be executed **in order**. Later cells lean on bindings from
  earlier ones.
- Code that intentionally *fails* to compile is shown as a comment, together with the
  error you would get. Rust's diagnostics are teaching material in this track, not an
  obstacle to it.
- The prose is terse and the facts are dense. This is a reference you will come back
  to, not a novel you read once.

## The one idea to internalise

Rust is **expression-oriented**. `if`, `match`, `loop` and a bare `{ ... }` block are
all expressions, each with a type, each usable anywhere a value is expected. That is
why `if` replaces the ternary operator, why a `loop` can `break` with a value, and why
the arms of an `if` or a `match` must agree on a type — the result has to live
somewhere before it is used.

The payoff is a style of code that *composes*. Instead of a sequence of statements
mutating shared variables, you get a chain of expressions, each producing the input
for the next. Where Python and C++ hand you statements and a `switch` that falls
through, Rust hands you values and a `match` that the compiler proves complete.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. `if` Is an Expression](#1-if-is-an-expression)
- [2. `else if`, `else` and Chains](#2-else-if-else-and-chains)
- [3. There Is No Truthiness](#3-there-is-no-truthiness)
- [4. `loop` and `break` with a Value](#4-loop-and-break-with-a-value)
- [5. `continue` and Labelled Loops](#5-continue-and-labelled-loops)
- [6. `while` and `while let`](#6-while-and-while-let)
- [7. `for` and Ranges](#7-for-and-ranges)
- [8. Iterating: by Value, by Reference, Mutably](#8-iterating-by-value-by-reference-mutably)
- [9. `match`: The Better `switch`](#9-match-the-better-switch)
- [10. Patterns in Depth](#10-patterns-in-depth)
- [11. Guards, `@` Bindings and `matches!`](#11-guards-bindings-and-matches)
- [12. `if let` and `let ... else`](#12-if-let-and-let-else)
- [13. Blocks as Expressions, and Diverging Expressions](#13-blocks-as-expressions-and-diverging-expressions)
- [14. Control Flow over `Option` and `Result`](#14-control-flow-over-option-and-result)
- [15. Reading the Compiler on Control Flow](#15-reading-the-compiler-on-control-flow)
- [16. Cheat Sheet](#16-cheat-sheet)

The sections are cumulative: each assumes the ones above it. If you are returning to
look something up, the [cheat sheet](#16-cheat-sheet) at the end is the fastest route
back to a specific piece of syntax.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** Jupyter kernel, the same one chapter 01 used. The
essentials are repeated here because every convention below depends on them.

Ordinarily you write a Rust program as a file containing items — chiefly `fn main()` —
and compile the whole thing. evcxr does something different: it wraps whatever you
type into a generated program, compiles it incrementally, and keeps the results alive
for the next cell. In effect you are typing the *body* of `main`, with the compiler
permanently running behind you.

### What that means in practice

- **No `fn main` in these cells.** The kernel supplies it.
- **State persists across cells.** A binding introduced in cell 3 is still in scope in
  cell 12; a `fn`, `struct`, `const` or `use` defined anywhere is visible to every
  later cell. **Run the notebook top to bottom.**
- **Items are order-independent; statements are not.** Two function definitions in one
  cell may call each other freely, in either order. Two `let` statements may not use
  each other's names before they have actually run.
- **There is one namespace per notebook.** Two functions with the same name in
  different sections collide, exactly as they would in one source file. When you write
  your own experiments, keep names distinct.
- **Re-running a cell that defines an item re-defines it.** If the kernel complains
  that something is already defined, *Restart* and run from the top. Restarting is
  cheap and always correct.
- **A failing cell does not poison the kernel.** You get a compile error or a panic
  for that cell alone. Fix it and re-run just that cell.
- **`println!` writes to stdout**; `eprintln!` and `dbg!` write to stderr.

### The two habits worth carrying over

**Restart the kernel before investigating** anything that behaves inexplicably: stale
state is the most common cause of "impossible" behaviour, and the diagnosis is free.

**Read the compiler's message before changing the code.** This chapter leans on that
habit especially hard, because `match` and `if let` generate some of the most helpful
diagnostics in the language — including, quite often, the exact fix.

If you have not read chapter 01, its §0 covers the same ground with the reasoning
attached.


In [ ]:
// 0.1 Proof that state persists: this binding is visible in every later cell,
// and a `fn` defined anywhere in the notebook is callable from anywhere.
let kernel_marker = "state persists across cells";
println!("{kernel_marker}");


## 1. `if` Is an Expression

The first difference from C, C++ and Java is superficial: **no parentheses around the
condition**. The second is superficial but rigid: **the braces are mandatory**, even
for a single statement. Write `if (x > 0)` and it compiles, but rustc warns
`unnecessary parentheses around `if` condition` — the language wants you to see the
condition as an expression, not as a C-style test.

The substantive difference is that `if` **produces a value**. There is no ternary
operator `?:` in Rust, and it would be redundant: `if cond { a } else { b }` already
*is* an expression, and it reads better.

### The syntax, precisely

| Form | Type |
|---|---|
| `if cond { ... }` | `()` — always |
| `if cond { a } else { b }` | type of `a` and `b`, which must be identical |
| `if cond { a } else if c { b } else { d }` | type of `a`, `b` and `d` |

That first row is worth sitting with. An `if` with no `else` *always* has type `()`,
no matter what its body does, because the branch might not run and there is nothing
else for the expression to be worth. So `let x = if cond { 1 };` is an error: when the
condition is false there is no value to bind.


In [ ]:
// 1.1 `if`: a `bool` condition, no parentheses, and a mandatory block.

let temperature = 21;

// The condition is not parenthesised. The body is a block, and the braces are
// not optional even for a single statement.
if temperature > 30 {
    println!("scorching");
}

if temperature > 18 {
    println!("pleasant ({temperature} °C)");
}

// Because the body is a block, it holds as many statements as you like.
let mut label = String::new();
if temperature < 25 {
    label.push_str("mild");
    label.push('!');
}
println!("label = {label}");

// There is no `elseif` and no separate `else if` keyword -- it is `else`
// followed by a block that happens to contain one `if`. See §2.
//
// And the condition really must be a `bool`:
//
//     if temperature { }                 // expected `bool`, found integer
//     if (temperature > 18) { }          // compiles, warns `unused_parens`
let is_warm = temperature > 18;
if is_warm {
    println!("is_warm = {is_warm}");
}


### `if` produces a value

This is what replaces the ternary operator. The whole `if`/`else` is an expression, so
its value is the value of whichever branch ran, and it can appear anywhere a value
can: on the right of a `let`, inside a function call, or as the tail expression of a
function body.


In [ ]:
// 1.2 `if` as an expression: the ternary operator, spelled out.

let n = 7;

// Python's `"even" if n % 2 == 0 else "odd"` and C's
// `n % 2 == 0 ? "even" : "odd"` are both written this way.
let parity = if n % 2 == 0 { "even" } else { "odd" };
println!("{n} is {parity}");

// Chained, it replaces a nested ternary.
let sign_word = if n < 0 {
    "negative"
} else if n == 0 {
    "zero"
} else {
    "positive"
};
println!("sign_word = {sign_word}");

// It composes with anything that takes a value, including another call.
println!("nested in a call: {}", if n > 5 { "big" } else { "small" });

// And it can be the tail expression of a function body -- no `return` needed,
// because the `if` is the value of the block (chapter 03).
fn abs_via_if(x: i32) -> i32 {
    if x < 0 { -x } else { x }
}
println!("abs_via_if(-9) = {}", abs_via_if(-9));

// Because it is an expression, an arm can be a whole block.
let score = 82;
let bucket = if score >= 80 {
    let margin = score - 80;
    format!("B, {margin} over the line")
} else {
    "below B".to_string()
};
println!("bucket = {bucket}");


### The two type rules

An `if` expression has to have **one** type before it runs — the compiler needs a slot
to put the result in — so the two arms must agree. And an `if` with no `else` can
produce nothing, so its type is `()`. Both rules produce errors that name the problem
directly, and both are worth seeing once.


In [ ]:
// 1.3 The two type rules, and the diagnostics that enforce them.

// Rule 1: both arms must have the SAME type.
//
//     let flag = true;
//     let x = if flag { 1 } else { "a" };
//     error[E0308]: `if` and `else` have incompatible types
//       |
//       |     let x = if c { 1 } else { "a" };
//       |                    -          ^^^ expected integer, found `&str`
//       |                    |
//       |                    expected because of this
//
// The fix is to make both arms agree -- convert, or return a common type:
let flag = true;
let x = if flag { "1".to_string() } else { "a".to_string() };
println!("both arms String: {x}");

// Rule 2: an `if` with no `else` has type `()`. That is fine as a statement,
// and the `()` is real and printable:
let unit = if n > 0 { println!("  n is positive"); };
println!("an `if` without `else` yields {unit:?}");

// But it cannot be bound as a value:
//
//     let bad = if n > 0 { n };
//     error[E0317]: `if` may be missing an `else` clause
//       |
//       |     let bad = if n > 0 { n };
//       |               ^^^^^^^^^^^-^^
//       |               |          |
//       |               |          found here
//       |               expected integer, found `()`
//       |
//       = note: `if` expressions without `else` evaluate to `()`
//       = help: consider adding an `else` block that evaluates to the expected type
//
// The fix is the obvious one, and the compiler even suggests it:
let good = if n > 0 { n } else { 0 };
println!("good = {good}");


## 2. `else if`, `else` and Chains

Rust has neither an `elseif` nor an `else if` keyword. What you write as `else if` is
literally `else` followed by a block, whose only statement is another `if` expression.
It reads like Python's `elif` and behaves like it: the first branch whose condition
holds wins, and the rest are never evaluated.

### Chains versus early returns

A chain tests one value against several mutually exclusive conditions, and produces one
of several results. It is the right shape when the cases really do belong together —
a grade boundary, a status mapping, a size class.

The failure mode is **ordering**. Because the first true condition wins, a broad test
placed before a narrow one makes the narrow one dead code. The compiler cannot help:
unlike `match`, an `if` chain is not analysed for reachability, because the conditions
are arbitrary expressions rather than patterns.


In [ ]:
// 2.1 An `else if` chain: one value tested against several conditions.

fn grade(score: u32) -> char {
    if score >= 90 {
        'A'
    } else if score >= 80 {
        'B'
    } else if score >= 70 {
        'C'
    } else if score >= 60 {
        'D'
    } else {
        'F'
    }
}

for score in [95u32, 90, 85, 72, 60, 59, 0] {
    println!("  {score:>3} -> {}", grade(score));
}

// Each arm is a block, so an arm can compute before producing its value.
fn describe(score: u32) -> String {
    if score > 100 {
        format!("impossible ({score})")
    } else if score == 100 {
        "perfect".to_string()
    } else if score >= 60 {
        format!("passed with {}", score - 60)
    } else {
        format!("failed, {} short", 60 - score)
    }
}
println!("{}", describe(84));
println!("{}", describe(41));


### When order stops being free

Because the first true condition wins, reordering a chain can quietly change the
meaning of the program. The compiler will not tell you: it cannot know which ordering
you intended, and it does not analyse `if` chains for reachability the way it analyses
`match` arms. This is the one place in this chapter where a bug is easy to write and
invisible to the type checker.


In [ ]:
// 2.2 Ordering is part of the logic, and the compiler will not check it.

fn classify(n: i32) -> &'static str {
    if n < 0 {
        "negative"
    } else if n == 0 {
        "zero"
    } else if n < 10 {
        "small positive"
    } else {
        "large positive"
    }
}
for value in [-5, 0, 3, 42] {
    println!("  classify({value}) = {}", classify(value));
}

// Same conditions, broadest test first. Nothing warns -- `n == 0` is simply
// unreachable, and the middle arm of the correct version is gone.
fn classify_broadly(n: i32) -> &'static str {
    if n >= 0 {
        "non-negative"
    } else if n == 0 {
        "zero"      // dead: every 0 already matched `n >= 0`
    } else if n > 0 {
        "positive"  // dead: every positive already matched `n >= 0`
    } else {
        "negative"
    }
}
for value in [-5, 0, 3, 42] {
    println!("  classify_broadly({value}) = {}", classify_broadly(value));
}

// Contrast with `match`, which does warn about this (§9, §10): patterns can
// be analysed for overlap and unreachability, whereas `if` conditions are
// arbitrary expressions and cannot be.
println!("{}", classify_broadly(0));


## 3. There Is No Truthiness

In Python, `if []:`, `if 0:` and `if "":` all take the false branch. In C and C++,
`if (0)`, `if (NULL)` and `if (0.0)` do. Rust has **no truthiness at all**: an `if`
condition must have type `bool`, and nothing is implicitly convertible to it.

This is one of the clearest places where the language trades convenience for
certainty. Every condition becomes an explicit, named question, and every question is
answered in exactly one way. In exchange:

- `if (x = 5)` — assignment used as a condition, a source of decades of C bugs —
  is a **type error**, not a warning.
- `if 1` is not "always true", it is a compile error.
- There is no integer `0`/`1` acting as `false`/`true`, so no `if (flags & MASK)`
  ambiguity and no `if (strcmp(a, b))` inversion.
- A function returning a `bool` cannot accidentally return an error code that gets
  mistaken for success, because the types do not line up.


In [ ]:
// 3.1 The two conditions that do not compile.

// A number is not a condition:
//
//     if 1 { println!("hello"); }
//     error[E0308]: mismatched types
//      --> probe.rs:6:8
//       |
//     6 |     if 1 { println!("nope"); }
//       |        ^ expected `bool`, found integer

// And assignment is not comparison -- rustc even hands you the fix:
//
//     let mut y = 0;
//     if y = 5 { }
//     error[E0308]: mismatched types
//       |
//       |     if y = 5 { println!("nope"); }
//       |        ^^^^^ expected `bool`, found `()`
//       |
//     help: you might have meant to compare for equality
//       |
//       |     if y == 5 { println!("nope"); }
//       |           +

// Meanwhile, zero, empty strings and empty containers are all perfectly good
// *values*. They are simply not `false`. You ask the question explicitly.
let zero = 0;
let empty = "";
let list: Vec<i32> = Vec::new();
let answer = true;
println!("zero == 0           -> {}", zero == 0);
println!("empty.is_empty()    -> {}", empty.is_empty());
println!("list.is_empty()     -> {}", list.is_empty());
println!("just `answer`       -> {answer}");
println!("u8::from(answer)    -> {}", u8::from(answer));


### Writing conditions that would have been truthy elsewhere

The translation is mechanical once you see the pattern, and it is usually *more*
informative than the original: `if s: }` becomes `if !s.is_empty()`, which says what
was actually being asked.

| Idiom elsewhere | In Rust |
|---|---|
| `if items:` | `if !items.is_empty()` |
| `if not items:` | `if items.is_empty()` |
| `if name:` / `if s:` | `if !name.is_empty()` |
| `if count:` / `if i:` | `if count != 0` |
| `if ptr:` / `if obj:` | no equivalent — there is no null; absence is `Option` (§14) |
| `if x is None:` | `if opt.is_none()` |
| `if x is not None:` | `if opt.is_some()`, or `if let Some(v) = opt` |
| `if a and b:` | `if a && b` |
| `if not ok:` | `if !ok` |

Note the last two: `&&` and `||` still **short-circuit**, so the guard order matters
exactly as much as it did before — and for once, that is good news, because it means
`if list.is_empty() || list[0] > 0` is a safe idiom rather than an out-of-bounds panic.


In [ ]:
// 3.2 Writing the condition explicitly, and naming it when it helps.

// A `bool` is a first-class value, so a condition can be given a name. This
// is often the clearest fix for a long chain -- and it is free.
let is_weekend = false;
let is_holiday = true;
let works_today = !is_weekend && !is_holiday;
println!("works_today = {works_today}");

// Short-circuiting is what makes the "check the container first" idiom safe.
// `list[0]` is never evaluated when the left operand is false.
let maybe: Vec<i32> = Vec::new();
if maybe.is_empty() || maybe[0] > 0 {
    println!("guarded access: the index was never evaluated");
}

// Compare with the same logic in C++, where `||` is the same but `[0]` on an
// empty vector is undefined behaviour that the compiler will not catch:
//
//     if (v.empty() || v[0] > 0) { ... }      // safe only because of the order
//
// In Rust the order still matters, but `v[0]` on an empty `Vec` panics with a
// readable message instead of reading arbitrary memory, and the borrow checker
// stops you from keeping a reference across the mutation that invalidates it.

// `bool` values compose, and `matches!` (§11) turns a pattern test into one:
let tokens = ["id", "42", "-", "7"];
let numeric = tokens.iter().filter(|t| matches!(t.parse::<i32>(), Ok(_))).count();
println!("numeric tokens = {numeric}");


## 4. `loop` and `break` with a Value

Rust has three loops, and they are not interchangeable:

| Loop | Ends when | Expression value |
|---|---|---|
| `loop` | you `break` | whatever you `break` with |
| `while` | the condition becomes false | `()` |
| `for` | the iterator is exhausted | `()` |

`loop` is the unconditional one. The only ways out are `break`, `return`, or a panic —
so it is the honest spelling of "keep going until something happens", and the compiler
knows a `loop` with no `break` never finishes.

### Why `break` with a value matters

Because `loop` is the *only* loop that can `break` with a value, it is the idiomatic
way to write a search. In C or Python you would need a sentinel (`-1`), a flag, or a
variable declared before the loop and mutated inside it. In Rust the result of the
search *is* the value of the loop, and the compiler proves that the variable is
initialised — there is no "found" flag, no sentinel that could collide with a real
result, and no way to read a half-computed answer.


In [ ]:
// 4.1 `loop` is unconditional; `break` is the only planned exit.

let mut attempts = 0;
loop {
    attempts += 1;
    // Pretend there is work here that might eventually succeed.
    if attempts == 3 {
        println!("succeeded after {attempts} attempts");
        break;
    }
}
println!("the loop exited cleanly");

// A `loop` with no `break` has type `!` (never), which is compatible with any
// type. That is what lets it stand in for a value that is never produced:
//
//     fn serve_forever() -> i32 {
//         loop {
//             // handle a request ...
//         }
//     }
//
// The body never returns a value, so nothing is needed after the loop, and the
// return type is a lie the compiler accepts because the function never falls
// out of the bottom. See §13 for divergence.

// `while` and `for` cannot break with a value -- they yield `()` -- which is
// exactly why the search idiom below uses `loop`.
let mut candidate = 1u32;
let first_cube_over_1000 = loop {
    candidate += 1;
    if candidate * candidate * candidate > 1000 {
        break candidate;
    }
};
println!("first cube over 1000 = {first_cube_over_1000}");


### `break value` — the search idiom

`break` accepts an expression, and every `break` in a `loop` must produce the same
type, because that type *is* the type of the loop. This turns "find the first thing
that satisfies a condition" into a single expression — with nowhere to forget to set a
flag, and with the compiler proving the result is always initialised before it is read.

A `loop` whose only exits carry values never falls through, so no path leaves the
binding unassigned. Compare with C, where `int found = -1;` has to encode "nothing
found" *inside* the result type, or with Python, where `result = None` needs an
`is not None` check afterwards.


In [ ]:
// 4.2 `break value` in a real search, and the type rule that comes with it.

fn first_square_over(limit: u32, ceiling: u32) -> Option<u32> {
    let mut n = 1;
    loop {
        let square = n * n;
        if square > ceiling {
            // Two exits, both carrying a `u32`... except this one carries the
            // type the *function* returns. A `break` value must match the
            // loop's other `break` values, not the function's return type.
            break None;
        }
        if square >= limit {
            break Some(n);
        }
        n += 1;
    }
}
println!("{:?}", first_square_over(50, 1000));
println!("{:?}", first_square_over(50, 10));

// Both `break`s above produce `Option<u32>`, so the loop is an
// `Option<u32>`, and the `let` binds it directly. A `loop` never falls
// through, so the binding is provably initialised.

// `break` outside a loop is one of the few clearly impossible things to write:
//
//     break 5;
//     error[E0268]: `break` outside of a loop or labeled block
//       |
//       |     break 5;
//       |     ^^^^^^^ cannot break outside of a loop or labeled block
//
// Note "or labeled block" -- §5.2 covers breaking out of a plain `{ }`.

// A `loop { ... break; }` is also how you write an accumulator without a
// sentinel, because the accumulator is checked by the same rule:
let total: i64 = loop {
    let mut acc = 0i64;
    for k in 1..=5 {
        acc += k;
    }
    break acc;
};
println!("total = {total}");


## 5. `continue` and Labelled Loops

`continue` abandons the current iteration and starts the next one. It exists so that
the *common* case can be stated first and the *interesting* case last, which is
usually the more readable order.

Its cost is that it makes the loop harder to reason about: code after a `continue` in
the same iteration does not run, and a `continue` placed before a counter increment
silently skips the increment. Rust will not warn, because it cannot know which
placement you intended — but it *will* refuse to compile a `for` loop whose iterator
you try to advance by hand, which removes the classic C pitfall entirely.

### Loops are labelled, and labels are type-checked

Nested loops need to say *which* loop they mean, and Rust does this with a label: a
tick followed by a name, written before the loop. `break 'outer` and `continue
'outer` then act on the named loop rather than the innermost one — the same job as
`goto`, but scoped to a loop that the compiler can see, and impossible to point at a
label that does not exist.


In [ ]:
// 5.1 `continue`: skip the rest of this iteration.

// The guard-first style. In Python and C++ the same thing is often written as
// a big `if` wrapping the body, which nests one level deeper for every guard.
let mut evens = Vec::new();
for n in 1..=12 {
    if n % 2 != 0 {
        continue;   // odd numbers get no further
    }
    evens.push(n);
}
println!("evens = {evens:?}");

// Where `continue` bites in C: incrementing the counter happens in the `for`
// header, so a `continue` cannot skip it. Rust's `for` hides the increment
// entirely, so this class of bug cannot be written.
//
// The equivalent mistake is a hand-written `while` where the increment is at
// the bottom and a `continue` jumps over it:
let mut i = 0;
let mut kept = Vec::new();
while i < 6 {
    // If this were `i += 1; continue;` the loop would spin forever -- and it
    // really would, with no diagnostic. Be deliberate about this placement.
    i += 1;
    if i % 3 == 0 {
        continue;
    }
    kept.push(i);
}
println!("kept = {kept:?}");

// `continue` also works in `loop` and in `while let` (§6).
// `filter` is usually clearer than a loop plus `continue` when all you are
// doing is discarding values:
let odds: Vec<i32> = (1..=12).filter(|n| n % 2 != 0).collect();
println!("odds  = {odds:?}");


### Labelled loops: steering the outer loop

Without labels, breaking out of a nested search means a flag and a test in the inner
loop — or, in C, a `goto`. Rust's labels are a scoped, checked version of `goto`: they
name a loop, they are visible only inside it, and the compiler rejects a `break` or
`continue` that names nothing.

The syntax is a tick and a name before the loop keyword: `'outer: for ...`. A label is
not a variable and cannot be stored or passed; it exists only to be named by `break`
and `continue` inside that loop.


In [ ]:
// 5.2 Labels: `break 'name` and `continue 'name` reach the enclosing loop.

// Find the first pair whose sum exceeds a budget. Without labels this needs a
// `found` flag and a check in the inner loop after every iteration.
let values = [1, 2, 3, 4, 5, 6];
let budget = 10;
let mut found = None;

'search: for (i, a) in values.iter().enumerate() {
    for (j, b) in values.iter().enumerate() {
        if i >= j {
            continue;               // unlabelled: skips this inner iteration
        }
        if a + b > budget {
            found = Some((*a, *b));
            break 'search;          // labelled: leaves BOTH loops
        }
    }
    println!("  finished the inner loop for i={i}");  // not reached for i=4
}
println!("found = {found:?}");

// `continue 'label` jumps to the *next iteration of the named loop*, which is
// the way to skip the rest of the outer body from inside the inner one.
let mut pairs = Vec::new();
'outer: for x in 1..=3 {
    for y in 1..=3 {
        if y > x {
            continue 'outer;        // abandon the rest of this x entirely
        }
        pairs.push((x, y));
    }
}
// x=3 is the only outer iteration that runs to completion, so it is the only
// one that contributes its full row.
println!("pairs = {pairs:?}");

// A labelled `loop` breaks with a value, labels and all.
let mut m = 0;
let thirty = 'spin: loop {
    m += 1;
    if m >= 30 {
        break 'spin m * 10;
    }
};
println!("thirty = {thirty}");


### Labelled *blocks* — a scoped `goto` for early exits

A label can sit on a plain block as well as on a loop: `'blk: { ... break 'blk value; }`.
This is a way to give a name to an early exit from a straight-line piece of code,
without introducing a loop that never loops or a function that exists only to `return`.

It is worth knowing about because the error message for a stray `break` mentions it
(`break outside of a loop or labeled block`), and because it is the mechanism behind
`try` blocks and similar proposals. In ordinary code, a labelled loop keeps an early
exit readable; a labelled block is the same trick when there is no loop to label.


In [ ]:
// 5.3 A labelled block gives a name to an early exit -- no loop required.

// Without the label this needs a nested `if`, or a helper function whose only
// job is to `return` early. With it, the bail-out reads like a guard clause.
let candidate = 7;
// Note the `: String`. Every `break` value and the block's own tail expression
// have to be the *same* type, so the type is written once at the binding and
// the rest is checked against it. (The first draft of this cell mixed `&str`
// breaks with a `format!` tail and rustc gave `expected &str, found String` --
// a labelled block is an expression like any other, and one expression has one
// type.)
let verdict: String = 'blk: {
    if candidate < 0 {
        break 'blk String::from("rejected: negative");
    }
    if candidate % 2 == 0 {
        break 'blk String::from("rejected: even");
    }
    if candidate < 5 {
        break 'blk String::from("rejected: too small");
    }
    // No `break` down here: the last expression of the block is its value.
    format!("accepted: {} x 3 = {}", candidate, candidate * 3)
};
println!("verdict = {verdict}");

// The same rule applies to a labelled block over any type -- the tail must
// agree with every `break`:
//
//     let bad = 'b: {
//         if true { break 'b 1; }
//         "text"
//     };
//     error[E0308]: mismatched types
//        |
//        |         "text"
//        |         ^^^^^^ expected integer, found `&str`

// A labelled block is also the neatest way to scope a temporary:
let scaled = {
    let factor = 1.5_f64;
    (candidate as f64) * factor
};
println!("scaled = {scaled}");


## 6. `while` and `while let`

`while` is the loop for "keep going while a condition holds". It is the one to reach
for when the number of iterations is not known in advance and there is no container to
walk — a convergent numeric method, a retry budget, a state machine.

Its two structural gaps relative to C are deliberate. There is no `do { } while`; an
unconditional first pass is written as `loop { ... if !cond { break; } }`, which makes
the exit visible at the point where it happens. And there is no `for`-with-condition:
if you need an index, you maintain it explicitly and the compiler checks it.

### `while let` — looping until a pattern stops matching

`while let PATTERN = EXPRESSION { ... }` repeats as long as the pattern matches,
re-evaluating the expression each time. It is the standard way to drain a structure
whose members are returned as `Option`:

- `while let Some(top) = stack.pop()` — until the stack is empty.
- `while let Ok(line) = reader.read_line(&mut buf)` — until reading fails.
- `while let Some(x) = it.next()` — until the iterator is exhausted (though `for` is
  better whenever the iterator is the whole story).

The loop ends for the *only* reason the pattern allows it to: there is no separate
condition to get out of sync with the pattern.


In [ ]:
// 6.1 `while`: an explicit condition, and the `loop`-plus-`break` equivalent.

// A convergent calculation: keep halving until we are close enough. The
// number of iterations is not known in advance, which is exactly the case
// `while` is for.
let mut value = 100.0_f64;
let mut halvings = 0;
while value > 1.0 {
    value /= 2.0;
    halvings += 1;
}
println!("after {halvings} halvings: {value}");

// There is no `do { } while` in Rust. The unconditional-first-pass pattern is
// a `loop` with the test at the bottom, which puts the exit next to the
// reason for it.
let mut n = 1000;
let mut digits = 0;
loop {
    digits += 1;
    n /= 10;
    if n == 0 {
        break;
    }
}
println!("1000 has {digits} digits");

// The condition is a `bool` and nothing else (§3), so a `while` whose
// condition is accidentally `()` will not compile:
//
//     let mut k = 0;
//     while k = 1 { }        // expected `bool`, found `()`
//
// The idiomatic spellings of a counter loop, in descending order of
// preference:
let mut total = 0;
let mut i = 0;
while i < 5 {
    total += i;
    i += 1;
}
println!("while counter:      {total}");

let mut total2 = 0;
for i in 0..5 {
    total2 += i;
}
println!("for over a range:   {total2}");

let total3: i32 = (0..5).sum();
println!("iterator `sum`:     {total3}");


In [ ]:
// 6.2 `while let`: loop until a pattern stops matching.

// `Vec::pop` returns `Option<T>` -- `Some` while there is something left,
// `None` when the vector is empty. `while let` says "keep going while it
// matches", which removes the length check that could go out of sync.
let mut stack = vec![1, 2, 3];
let mut popped = Vec::new();
while let Some(top) = stack.pop() {
    popped.push(top);
}
println!("popped = {popped:?}, stack = {stack:?}");

// Any expression that yields `Option` works, and the loop re-evaluates it
// every time round -- this is a real call per iteration, not a cached value.
let mut budget = 3;
let mut spent = Vec::new();
while let Some(next) = if budget > 0 { budget -= 1; Some(budget) } else { None } {
    spent.push(next);
}
println!("spent = {spent:?}");

// `continue` inside `while let` skips the rest of the body but still
// re-evaluates the scrutinee, so no item is lost:
let mut mixed = vec![3, 1, 2];
let mut kept_odd = Vec::new();
while let Some(v) = mixed.pop() {
    if v % 2 == 0 {
        continue;
    }
    kept_odd.push(v);
}
println!("kept_odd = {kept_odd:?}");

// `Some(top)` is a refutable pattern: it can fail. An `if let` / `while let`
// is what you reach for when you expect it to fail eventually. A plain `let`
// with the same pattern would be, correctly, a compile error. See §12.


## 7. `for` and Ranges

Rust's `for` is not a counter loop with a header, an increment and a test. It is
`for PATTERN in ITERATOR`, and its body runs once per item the iterator yields. There
is no index to initialise, no bound to re-check and no increment to forget: the
iterator knows when it is done.

This is the single biggest day-to-day change coming from C or C++, where
`for (int i = 0; i < n; ++i)` is the default. In Rust the default is
`for item in collection`. `i < n` versus `i <= n`, an off-by-one in the increment, a
bound recomputed on every pass, a `vec.size()` called inside the condition — none of
those can happen, because none of those parts exist.

Ranges supply the cases where you really do want numbers. A range is just a value, and
a range of integers is an iterator.

| Range | Means | Iterates |
|---|---|---|
| `a..b` | half-open | `a`, `a+1`, … `b-1` |
| `a..=b` | inclusive | `a`, `a+1`, … `b` |
| `a..` | from `a` onwards | `a`, `a+1`, … — **not** usable in `for` (it never ends) |
| `..b` | up to `b` | not usable in `for` for the same reason |
| `..=b` | up to and including `b` | not usable in `for` |
| `..` | everything | used in patterns and slices, not in `for` |

The `for`-forbidden forms are exactly the ones with no upper bound, and rustc says so
plainly: `the range `..b` is not an iterator; you might want to use a different
iterator` with a suggestion. Open-ended ranges are still useful — as index bounds, as
slice ranges, and as `match` patterns (§10).


In [ ]:
// 7.1 Ranges are values, and a range of integers is an iterator.

// Half-open `a..b` excludes the end; inclusive `a..=b` includes it. Both are
// ordinary values -- a range can be stored, passed and printed.
let half_open = 1..5;
let inclusive = 1..=5;
println!("`1..5`  printed: {half_open:?}");   // Range<i32>, not a step-by-step
println!("`1..=5` printed: {inclusive:?}");

print!("1..5   ->");
for i in 1..5 {
    print!(" {i}");
}
println!();

print!("1..=5  ->");
for i in 1..=5 {
    print!(" {i}");
}
println!();

// Ranges are not iterators-to-be-consumed by hand, but the type has a name you
// will meet in compiler messages:
let r: std::ops::Range<i32> = 0..10;
println!("0..10 has {} elements", r.len());
println!("0..10 contains 9? {}", r.contains(&9));

// The adaptor methods chapter 06 covers properly, here to show that a range is
// just the start of a pipeline.
print!("(1..=5).rev()      ->");
for i in (1..=5).rev() {
    print!(" {i}");
}
println!();

print!("(1..10).step_by(3) ->");
for i in (1..10).step_by(3) {
    print!(" {i}");
}
println!();

print!("(0..5).map(x2)     ->");
for i in (0..5).map(|y| y * 2) {
    print!(" {i}");
}
println!();

print!("(0..3).chain(0..1) ->");
for i in (0..3).chain(0..1) {
    print!(" {i}");
}
println!();

// An empty range is a range, not an error. This is why `for` cannot go wrong
// at the boundary: `3..3` simply yields nothing.
let mut hits = 0;
for _ in 3..3 {
    hits += 1;
}
println!("3..3 yielded {hits} items");


### `for` walks any iterator

The loop itself does not care what it is walking. A range, a `Vec`, a slice, a `String`'s
characters, a map's entries, a file's lines, an adaptor chain — all of them are
iterators, and the same three-line loop handles each one.

Two things to notice in the examples below. First, `for x in collection` **consumes**
the collection, while `for x in &collection` borrows it; that difference is the subject
of §8 and of the whole of chapter 05. Second, `.enumerate()` is how you get an index
when you want one, and it is the right answer far more often than a manual counter.

Unlike the C family, `for` is also the *fastest* option: iterating a slice with `for`
compiles to the same code as the hand-written index loop, and the compiler is free to
vectorise it precisely because there is no aliasing index to reason about.


In [ ]:
// 7.2 `for` over the things you actually iterate.

// Over a range -- the replacement for `for (int i = 0; i < n; ++i)`.
let mut squares = Vec::new();
for n in 1..=5 {
    squares.push(n * n);
}
println!("squares = {squares:?}");

// Over a reference to a Vec, so the Vec survives the loop (§8).
let words = vec!["alpha", "beta", "gamma"];
let mut lengths = Vec::new();
for w in &words {
    lengths.push(w.len());
}
println!("lengths = {lengths:?} (words still usable: {words:?})");

// Over the characters of a string. Note `.chars()`, not `.bytes()` (§9 of
// chapter 04) -- and note that a `String` cannot be iterated directly:
//
//     for c in "hello" { }        // the compiler suggests `.chars()`
for c in "héllo".chars() {
    print!("[{c}]");
}
println!();

// Over pairs, and over an index plus a value.
let names = ["ada", "grace"];
let ages = [36, 45];
for (name, age) in names.iter().zip(ages.iter()) {
    println!("  {name} is {age}");
}

for (i, name) in names.iter().enumerate() {
    println!("  {i}: {name}");
}

// `_` as a pattern discards the item. `for _ in 0..3` means "three times".
let mut count = 0;
for _ in 0..3 {
    count += 1;
}
println!("ran {count} times");

// Ranges with a step and a direction compose with `for` unchanged.
print!("countdown:");
for n in (1..=5).rev() {
    print!(" {n}");
}
println!();

// Destructuring in the loop pattern works for any iterable of tuples:
let pairs = [(1, "one"), (2, "two")];
for (num, word) in pairs {
    println!("  {num} -> {word}");
}


## 8. Iterating: by Value, by Reference, Mutably

`for` has three shapes, and choosing between them is a preview of chapter 05. The
choice is spelled by what you put after `in`:

| You write | Each item is | The collection afterwards |
|---|---|---|
| `for x in collection` | the value itself (`T`) | **moved** — gone unless `T: Copy` |
| `for x in &collection` | a shared reference (`&T`) | still yours, unchanged |
| `for x in &mut collection` | a mutable reference (`&mut T`) | still yours, modified |

`for x in collection` is the one that surprises people. In Python and C++ a `for` loop
never destroys the thing it walks; in Rust, iterating *by value* consumes the
container, because handing out each element by value means relinquishing ownership of
it. The compiler will not let you use the container afterwards, and it says so:

```text
error[E0382]: use of moved value: `nums`
```

Two escape hatches matter. First, `&collection` is what you usually want, and it costs
nothing: a shared borrow. Second, **arrays of `Copy` types are unaffected**, because
copying an `i32` out of the array does not take it away from anyone — so
`for x in [1, 2, 3]` leaves the array perfectly usable. That difference between arrays
and `Vec`s is a recurring source of confusion and is simply a consequence of `Copy`.

To *change* elements while iterating, borrow mutably and dereference: `for x in &mut
nums { *x *= 10; }`. The `*x` is required — `x` is a `&mut T`, and assigning to `x`
would try to replace the reference rather than the referent.


In [ ]:
// 8.1 By value, by shared reference, by mutable reference.

// --- by value: consumes a Vec ---
let nums = vec![1, 2, 3, 4];
let mut total = 0;
for x in nums {
    //        ^^^^ `nums` is moved into the loop
    total += x;
}
println!("total = {total}");
// Uncomment to see the move:
//
//     println!("{nums:?}");
//     error[E0382]: borrow of moved value: `nums`
//        |
//        | let nums = vec![1, 2, 3, 4];
//        |     ---- move occurs because `nums` has type `Vec<i32>`,
//        |          which does not implement the `Copy` trait
//        | for x in nums {
//        |          ---- `nums` moved due to this implicit call to
//        |               `.into_iter()`
//        | println!("{nums:?}");
//        |           ^^^^ value borrowed here after move
//        |
//     help: consider iterating over a slice of the collection to
//           preserve ownership of the collection
//        |
//        | for x in &nums {
//        |          +

// --- by shared reference: the Vec survives ---
let nums = vec![10, 20, 30];
let mut sum_ref = 0;
for x in &nums {
    //  ^ each `x` is `&i32`
    sum_ref += *x;                    // `*x` reads through the reference
}
println!("sum_ref = {sum_ref}, nums still here: {nums:?}");

// `nums.iter()` is the explicit spelling of the same thing, and it is what the
// adaptor methods produce, so you will see it often.
let doubled: Vec<i32> = nums.iter().map(|x| x * 2).collect();
println!("doubled = {doubled:?}");

// --- by mutable reference: modify in place ---
let mut values = vec![1, 2, 3];
for x in &mut values {
    *x *= 100;                        // the `*` is required
}
println!("after &mut: {values:?}");

// A wrong version, for contrast:
//
//     for x in &mut values { x *= 100; }
//     error[E0368]: binary assignment operation `*=` cannot be applied
//         to type `&mut {integer}`
//     note: `{integer}` can be used as the type of a `&mut` reference, but
//           the dereference assigns to the reference instead
//
// `x *= 100` would try to multiply the *reference* by 100, which is not a
// thing. Deref first: `*x *= 100`.

// --- arrays of Copy types: iterating by value does NOT move them ---
let arr = [1u8, 2, 3];
let mut arr_total = 0;
for b in arr {
    arr_total += u8::from(b);
}
println!("arr_total = {arr_total}, arr still usable: {arr:?}");

// The same expression over a Vec would have moved it. The difference is that
// `u8` is `Copy`, so `arr` was copied into the loop rather than handed over.


## 9. `match`: The Better `switch`

`match` is the closest thing Rust has to `switch`, and it is a different construct in
three ways that all pull in the same direction.

**It is exhaustive.** Every possible value of the scrutinee must be covered by some
arm. The compiler checks this, which turns "I forgot the `default` case" from a
production bug into a build failure.

**It is an expression.** Like `if`, a `match` produces a value, so it can be bound
with `let`, returned from a function, or nested inside another `match`.

**There is no fall-through.** Each arm is a complete branch, and only one arm runs.
That eliminates the C and Java bug class where a missing `break` silently executes the
next case.

There is also no `case`, no `default`, no `:` after the pattern, and no `break`. The
syntax is `PATTERN => EXPRESSION`, with **commas between arms**.

### Patterns, not values

The arm's left-hand side is a *pattern*, not an expression to compare against. That is
the real difference from `switch`: `match` is not asking "does this equal that?" — it
is asking "does this value have this shape, and if so, what are its parts?" Integer
literals happen to be patterns that mean equality, but the same arm list can bind
variables, destructure tuples and structs, test ranges, and combine cases with `|`.

The compiler reads those patterns too, so it will tell you about arms that can never
run and about sub-cases you have not covered. Nothing in Python or C++ does that.


In [ ]:
// 9.1 `match` on integers: literal arms, or-patterns, range arms, `_`.

fn status_line(code: u16) -> &'static str {
    match code {
        200          => "OK",
        301 | 302    => "redirected",       // `|` is an or-pattern, not a bitwise or
        304          => "not modified",
        400..=499    => "client error",     // a range is a pattern
        500..=599    => "server error",
        _            => "something else",   // `_` matches anything, like `default`
    }
}
for code in [200u16, 302, 304, 404, 500, 999] {
    println!("  {code} -> {}", status_line(code));
}

// `match` is an expression, so it binds directly:
let code = 404;
let is_error = match code {
    400..=599 => true,
    _ => false,
};
println!("is_error = {is_error}");

// Arms may be blocks, which is how you do work before producing the value:
let level = match code {
    200 => "info".to_string(),
    c if c >= 500 => format!("error {c}"),
    _ => {
        let base = code / 100;
        format!("class {base}xx")
    }
};
println!("level = {level}");

// Patterns can bind, and the binding is shadowed per-arm. `c` above was such a
// binding. Here the arm tests a range and names the value in one step (§11):
let bucket = match code {
    n @ 200..=299 => format!("success ({n})"),
    n @ 400..=499 => format!("your fault ({n})"),
    n @ 500..=599 => format!("our fault ({n})"),
    n => format!("unclassified ({n})"),
};
println!("bucket = {bucket}");


### Exhaustiveness: the arm you forgot

Every value of the scrutinee must be matched. Since `i32` has about four billion
values, that means an integer `match` effectively always needs a `_` arm — and when
you leave it out, rustc tells you precisely which values are uncovered, in the form of
interval patterns you could paste back in.

```text
error[E0004]: non-exhaustive patterns: `i32::MIN..=0_i32` and `3_i32..=i32::MAX`
              not covered
 --> probe.rs:4:11
  |
4 |     match n { 1 => "one", 2 => "two" };
  |           ^ patterns `i32::MIN..=0_i32` and `3_i32..=i32::MAX` not covered
  |
  = note: the matched value is of type `i32`
help: ensure that all possible cases are being handled by adding a match arm with
      a guard or pattern `i32::MIN..=0_i32 | 3_i32..=i32::MAX`
  |
4 |     match n { 1 => "one", 2 => "two", i32::MIN..=0_i32 | 3_i32..=i32::MAX => todo!() };
  |                                      +++++++++++++++++++++++++++++++++++++++++++++++
```

Three things are worth taking from that message. The uncovered set is reported as
**ranges**, not as a hand-wave about "not all cases handled". The `help` block offers
literally pastable code. And `todo!()` appears in it — the compiler's suggested
placeholder compiles and panics at runtime, which is the right tool for "I believe
this arm is unreachable; make the compiler check me".

For an enum, exhaustiveness is a feature rather than a nag: add a variant and every
`match` that does not handle it stops compiling, which is how you find the code that
needs updating. That is the whole reason to prefer `match` over a chain of `if`s.


### Patterns beyond integers: `bool`, `char`, tuples

`match` works on any type. Three cases worth seeing up close, because each shows a
different kind of pattern:

- **`bool`** — the compiler wants both arms, so `match flag { true => …, false => … }`
  is exhaustive without a `_`. You rarely write it (an `if` is shorter), but it is
  legal everywhere and it makes `bool` a perfectly good `match` scrutinee.
- **`char`** — arms can be single characters in single quotes, and character ranges
  work as patterns: `'a'..='z'` and `'A'..='Z'`. This is the tidy way to write character
  classification.
- **tuples** — a pattern can be a tuple pattern, destructuring the scrutinee across the
  arms. `(0, 0)`, `(x, 0)`, `(0, _)` and `_` between them cover the whole plain, and the
  compiler verifies that. Variables in a tuple pattern bind the corresponding
  component, and the binding's type is exactly that component's type.

This is where `match` starts to look less like a `switch` and more like a small
pattern-matching language — which §10 continues.


In [ ]:
// 9.2 `match` on `bool`, `char` and tuples.

// --- bool: exhaustiveness needs both arms, no `_` required ---
let flag = true;
let label = match flag {
    true => "on",
    false => "off",
};
println!("label = {label}");

// --- char: single-character literals, and character ranges ---
fn letter_class(c: char) -> &'static str {
    match c {
        'a'..='z' => "lowercase",
        'A'..='Z' => "uppercase",
        '0'..='9' => "digit",
        ' ' | '\t' | '\n' => "whitespace",
        '.' | ',' | ';' | ':' => "punctuation",
        _ => "other",
    }
}
for c in ['q', 'Q', '7', ' ', ',', '£'] {
    println!("  {c:?} -> {}", letter_class(c));
}

// --- tuples: destructuring across the arms ---
fn quadrant(x: i32, y: i32) -> &'static str {
    match (x, y) {
        (0, 0) => "origin",
        (_, 0) => "on the x axis",
        (0, _) => "on the y axis",
        (a, b) if a > 0 && b > 0 => "first quadrant",
        (a, b) if a < 0 && b > 0 => "second quadrant",
        (a, b) if a < 0 && b < 0 => "third quadrant",
        _ => "fourth quadrant",
    }
}
for (x, y) in [(0, 0), (3, 0), (0, -2), (1, 1), (-1, 1), (-1, -1), (1, -1)] {
    println!("  ({x:>2}, {y:>2}) -> {}", quadrant(x, y));
}

// A tuple pattern also binds parts that the arm then uses, and the whole thing
// can be matched on values that came from somewhere else entirely:
let pair = (2, 5);
let relation = match pair {
    (a, b) if a == b => format!("{a} = {b}"),
    (a, b) if a < b => format!("{a} < {b}"),
    (a, b) => format!("{a} > {b}"),
};
println!("relation = {relation}");

// `match` on a tuple can replace a table lookup, and rustc checks that the
// combinations you listed are the ones you meant:
let movement = match (flag, pair.0) {
    (true, 0) => "north",
    (true, _) => "forward",
    (false, _) => "hold",
};
println!("movement = {movement}");


## 10. Patterns in Depth

Now that integer matching is out of the way, here is what a pattern can actually be.
Everything in this list is checked by the compiler, and everything in it can appear in
a `match` arm — many of the forms also work in `let`, in function parameters, and in
`for` loop variables, which is why patterns are worth learning properly.

| Pattern | Matches |
|---|---|
| `42`, `'x'`, `"lit"`, `true` | exactly that value |
| `1..=9` | any value in the range |
| `_` | anything, binding nothing |
| `name` | anything, binding it to `name` |
| `(a, b)` | a 2-tuple, binding both components |
| `((a, b), c)` | a nested tuple — patterns nest arbitrarily |
| `Some(x)`, `Ok(v)` | a variant, binding its payload (§14) |
| `a \| b` | either pattern |
| `n @ 1..=9` | a range, **and** bind the value to `n` |
| `[a, b]` | a 2-element slice or array |
| `[first, rest @ ..]` | a slice with at least one element; `rest` is the tail |
| `..` | "and the rest", in tuples, structs and slices |

Two rules matter more than the table. A **bare identifier is a binding, not a
comparison**: the arm `x => …` matches everything and names it `x`, so an arm that
looks like a variable is never an equality test — that is why constants and literals
exist for this purpose. And a **binding shadows**, so an arm can reuse a name that
already exists in scope without touching the outer one.

### Overlapping and unreachable arms

Because patterns *can* be analysed, the compiler does analyse them, and it reports
arms that no value can reach. It will not do this for `if` chains (§2) — there the
conditions are arbitrary expressions, and only you know the intent.

```text
warning: unreachable pattern
 --> probe.rs:5:9
  |
4 |         1..=9 => "low",
  |         ----- matches all the relevant values
5 |         1..=5 => "also low",
  |         ^^^^^ no value can reach this
  |
  = note: `#[warn(unreachable_patterns)]` (part of `#[warn(unused)]`) on by default
```

Note that it names the *earlier* arm that swallowed the later one. Ordering still
decides which arm wins; the warning merely tells you when the later arm is dead.


In [ ]:
// 10.1 Patterns in practice: nesting, or-patterns, `@`, and slices.

// Nested tuple pattern -- patterns nest as deeply as the data does.
fn sum_flat(p: ((i32, i32), i32)) -> i32 {
    match p {
        ((a, b), c) => a + b + c,
    }
}
println!("sum_flat = {}", sum_flat(((1, 2), 3)));

// `@` binds the whole value while the pattern is still a range (§11 has more).
let x = 2;
let sized = match x {
    v @ (1 | 2 | 3) => format!("small: {v}"),
    v => format!("big: {v}"),
};
println!("sized = {sized}");

// Slice patterns: length, position and tail all at once.
fn shape(s: &[i32]) -> String {
    match s {
        [] => "empty".to_string(),
        [only] => format!("one: {only}"),
        [first, second] => format!("two: {first}, {second}"),
        [first, rest @ ..] => format!("first {first}, then {} more", rest.len()),
    }
}
let v = vec![1, 2, 3, 4];
for cut in [0, 1, 2, 4] {
    println!("  shape(&v[..{cut}]) = {}", shape(&v[..cut]));
}

// `..` in a `let` pattern discards the rest. Note `_` is not needed -- `..`
// says "the remaining components, whatever they are".
let (a, ..) = (1, 2, 3);
let [b, .., c] = [1, 2, 3, 4];
println!("a = {a}, b = {b}, c = {c}");

// The pitfall: a bare identifier in an arm is a BINDING, so it matches
// everything. Even if a variable of that name already exists outside, the arm
// shadows it and matches -- this is not a comparison:
let wanted = 5;
let surprise = match 99 {
    wanted => format!("bound `wanted` to {wanted}"),   // matches, binds 99
};
println!("surprise = {surprise}; the outer `wanted` is still {wanted}");

// To compare against a `const`, use the constant itself. A constant used as a
// pattern is a comparison, because it is not an identifier binding:
const TARGET: i32 = 5;
let honest = match 99 {
    TARGET => "equal to TARGET",
    _ => "not TARGET",
};
println!("honest = {honest} (TARGET is {TARGET})");

// And `_` is the deliberate discard, with no name to shadow:
let muted = match 42 {
    _ => "anything at all",
};
println!("muted = {muted}");


## 11. Guards, `@` Bindings and `matches!`

Three small features that make patterns considerably more expressive.

### Guards

An arm may carry an extra condition after the pattern: `PATTERN if CONDITION => …`.
The pattern must match **and** the condition must hold. This is how you attach a test
that cannot be expressed as a pattern — anything that needs arithmetic, a comparison
between two bound values, or a call.

A guard that fails does not end the `match`: control falls through to the *next* arm,
as if the pattern had not matched. That is the opposite of a `break`, and it is what
makes guards safe to use as extra filters. Guards also count towards exhaustiveness in
the sense that the compiler assumes they may fail, so an arm with a guard does not
cover the cases its pattern alone would cover.

### `@` bindings

`@` lets a pattern both match *and* name the value: `n @ 1..=9` matches a value in
1–9 and binds it to `n`. It is the answer to the common log message "value {n} is in
range 1..=9", where you want the test and the value without writing the range twice.

### `matches!`

`matches!(EXPR, PATTERN)` evaluates to a `bool`: `true` if the expression matches the
pattern. It is the bridge between the pattern language and ordinary boolean logic, and
it is the idiomatic way to ask "is this the right shape?" inside an `if` or a
`filter`. It also accepts guards: `matches!(x, Some(n) if n > 0)`.


In [ ]:
// 11.1 Guards and `@` bindings.

// A guard adds a condition the pattern cannot express. Here the pattern binds
// both components and the guard compares them -- impossible with a pattern.
fn classify_pair(a: i32, b: i32) -> &'static str {
    match (a, b) {
        (0, 0) => "both zero",
        (x, y) if x == y => "equal",
        (x, y) if x * y < 0 => "opposite signs",
        (_, _) => "mixed",
    }
}
for (a, b) in [(0, 0), (4, 4), (-3, 5), (1, 2)] {
    println!("  ({a}, {b}) -> {}", classify_pair(a, b));
}

// A failing guard falls through to the NEXT ARM, it does not abort the match.
// Watch `n = 3` skip the first arm and land on the second.
let n = 3;
let verdict = match n {
    x if x > 10 => format!("{x} is large"),
    x if x > 0 => format!("{x} is small and positive"),  // reached for n = 3
    x => format!("{x} is not positive"),
};
println!("verdict = {verdict}");

// `@` matches AND names. The range appears once, and the value is available.
let banded = match 7 {
    n @ 1..=5 => format!("{n} is low"),
    n @ 6..=10 => format!("{n} is mid"),
    n @ 11..=99 => format!("{n} is high"),
    n => format!("{n} is out of band"),
};
println!("banded = {banded}");

// `@` also works around or-patterns, giving one name to several shapes:
for c in ['a', 'Z', '5'] {
    let kind = match c {
        letter @ ('a'..='z' | 'A'..='Z') => format!("{letter} is a letter"),
        digit @ '0'..='9' => format!("{digit} is a digit"),
        other => format!("{other} is something else"),
    };
    println!("  {kind}");
}

// A guard can call a function, which no pattern can do:
fn is_prime(k: u32) -> bool {
    if k < 2 {
        return false;
    }
    let mut d = 2;
    while d * d <= k {
        if k % d == 0 {
            return false;
        }
        d += 1;
    }
    true
}
let prime_status = match 29u32 {
    k if is_prime(k) => format!("{k} is prime"),
    k => format!("{k} is not prime"),
};
println!("prime_status = {prime_status}");


In [ ]:
// 11.2 `matches!`: a `bool` from a pattern.

// Basic shape tests. Note the turbofish in the third line: `None` alone does
// not say what type of `Option` it is, so we name it.
println!("matches!(3, 1..=5)                        = {}", matches!(3, 1..=5));
println!("matches!(9, 1..=5)                        = {}", matches!(9, 1..=5));
println!("matches!(Some(3), Some(x) if x > 1)       = {}", matches!(Some(3), Some(x) if x > 1));
println!("matches!(None::<i32>, Some(_))            = {}", matches!(None::<i32>, Some(_)));

// It fits where a `bool` fits -- in an `if`. The type has to be wide enough for
// the range though: `200..=299` in an `Option<u8>` is a "literal out of range"
// error, because `u8` stops at 255.
let status: Option<u16> = Some(204);
if matches!(status, Some(200..=299)) {
    println!("status is in the 2xx class");
}

// ... and in a suite of comparisons, which is where it really earns its keep.
// Note the `&` before `t`: we are matching a `&str` element from `iter()`, and
// `matches!` does not peel the reference off for us.
let tokens = ["id", "42", "-7", "3.5", "true"];
let ints = tokens.iter().filter(|t| matches!(t.parse::<i32>(), Ok(_))).count();
let floats = tokens.iter().filter(|t| matches!(t.parse::<f64>(), Ok(_))).count();
let bools = tokens.iter().filter(|t| matches!(t.parse::<bool>(), Ok(_))).count();
println!("ints = {ints}, floats = {floats}, bools = {bools}");

// `matches!` is also the cleanest way to test a discriminated union of
// shapes without writing a full `match` that produces `()`:
let bounded = matches!(255u8, u8::MAX);
println!("bounded = {bounded}");

// For comparison: `is_some`/`is_none`/`is_ok`/`is_err` cover the common cases
// and read better when there is no payload to inspect. `matches!` is for
// everything else.
let maybe: Option<i32> = None;
println!("is_none = {}, matches! = {}", maybe.is_none(), matches!(maybe, None));


## 12. `if let` and `let ... else`

A `match` with one interesting arm and a do-nothing `_` is a lot of ceremony:

``` rust
match config_file {
    Some(path) => load(path),
    None => {}          // nothing to do
}
```

`if let` is the shorthand for exactly that shape: **run this block if the pattern
matches, otherwise skip it**.

``` rust
if let Some(path) = config_file {
    load(path);
}
```

It is an ordinary `if`, so it takes an `else`, and the `else` branch may itself be an
`if let` — though nesting more than once or twice is a sign that a `match` would read
better.

The important caveat: **`if let` is not exhaustive and nothing checks that it is.**
Because there is no `else` requirement, adding a variant to an enum will not make your
`if let`s fail to compile the way it will make your `match`es fail. Reach for `if let`
when you genuinely want "handle this case, ignore the rest", and for `match` when you
want to be told that you have handled everything.

`while let PATTERN = EXPR` is the same idea applied to repetition, and §6 covered it.

### `let ... else` — bail out early

`if let` handles the *matched* case inline and moves on. The inverted shape — "give me
the payload or get out of this function" — is what `let ... else` is for:

``` rust
let Some(path) = config_file else {
    return None;        // this block MUST diverge
};
load(path)              // `path` is in scope from here on
```

The `else` block must **diverge**: it has to `return`, `continue`, `break` or panic.
It is never a value, which distinguishes it from `if let`, and it is not optional,
which distinguishes it from `if let`'s missing `else`. The payoff is that `path` is
bound for the rest of the enclosing scope, unindented — the guard-clause style that
otherwise needs an extra level of nesting.


In [ ]:
// 12.1 `if let`, with and without an `else`.

// The shape it replaces: a `match` whose only remaining arm does nothing.
let config_file: Option<&str> = Some("/etc/app.conf");
match config_file {
    Some(path) => println!("match version: loading {path}"),
    None => {}
}

// The same thing, said in one line.
if let Some(path) = config_file {
    println!("if let version: loading {path}");
}

// `if let` takes an `else`, and because it is an ordinary `if` there is no
// exhaustiveness requirement. `None` may simply be ignored:
if let Some(path) = config_file {
    println!("path exists: {path}");
} else {
    println!("no config file; using defaults");
}

// Chained: an `else` branch that is itself an `if let`. Beyond two of these,
// a `match` is clearer -- and only `match` will tell you that you missed one.
let code: Result<i32, &str> = Err("bad input");
if let Ok(n) = code {
    println!("parsed {n}");
} else if let Err(e) = code {
    println!("failed: {e}");
}

// The pattern can be as elaborate as any `match` pattern (§10). Only the
// first matching shape runs.
let point = (0, 7);
if let (0, y) = point {
    println!("on the y axis at y = {y}");
}

let slice: &[i32] = &[1, 2, 3];
if let [first, rest @ ..] = slice {
    println!("if let on a slice: first = {first}, {} more", rest.len());
}

// Because it is an `if`, it is also an expression -- though almost always a
// `()` one, since the two branches rarely agree on a type.
let described = if let Some(p) = config_file { p.len() } else { 0 };
println!("described = {described} characters");


In [ ]:
// 12.2 `let ... else`: get the payload or leave, without an extra indent.

// `else` must diverge: `return`, `continue`, `break`, or a panic. Here the
// function's return type is the thing that makes `return None` legal.
fn parse_first(text: &str) -> Option<i32> {
    let mut it = text.split(',');
    let Some(first) = it.next() else {
        return None;                 // no field at all
    };
    let Ok(n) = first.trim().parse::<i32>() else {
        return None;                 // field is not a number
    };
    Some(n)
}
println!("parse_first(\"12,b\") = {:?}", parse_first("12,b"));
println!("parse_first(\"b,12\") = {:?}", parse_first("b,12"));
println!("parse_first(\"\")     = {:?}", parse_first(""));

// The payoff is scope: after a `let ... else`, the binding is available for
// the rest of the enclosing block with no extra nesting.
fn strict(text: &str) -> i32 {
    let Ok(n) = text.trim().parse::<i32>() else {
        return -1;
    };
    n * 2                              // `n` is right here, no `if` around it
}
println!("strict(\" 21 \") = {} (21 doubled)", strict(" 21 "));
println!("strict(\"x\")    = {} (bail-out)", strict("x"));

// A non-diverging `else` is a compile error, and the message says why:
//
//     let Some(x) = maybe else { };
//     error[E0308]: `else` clause of `let...else` does not diverge
//       |
//       |     let Some(x) = maybe else { };
//       |                             ^^^ expected `!`, found `()`
//       |
//       = note: `else` clause of `let...else` must diverge
//       = help: try adding a diverging expression, such as `return` or `panic!(..)`

// `continue` works too, which makes `let ... else` a filter inside a loop:
let mut kept = Vec::new();
for word in ["1", "two", "3", "four"] {
    let Ok(v) = word.parse::<i32>() else {
        continue;
    };
    kept.push(v);
}
println!("kept = {kept:?}");

// `break` is allowed as well, so this also works inside a `loop` that wants to
// stop early rather than skip. Nothing here is special-cased: the requirement
// is only that the `else` block never produces a value.


### Bonus: let-chains

This one is newer than the rest of the chapter and only compiles in **edition 2024**
and later. Two conditions can now be joined with `&&` inside a single `if`, including
`if let` and `while let`:

``` rust
if let Some((name, level)) = cfg && level > 2 {
    println!("{name} at level {level}");
}
```

Before this, the same logic needed either a nested `if` or a `match` with a guard. The
chain short-circuits in the obvious way: if the first pattern fails, the later
conditions are never evaluated, so the bindings they depend on are always initialised
by the time they run. That is checked by the compiler, not by convention.

Because edition is a per-crate setting, this is exactly the kind of feature that makes
an "old" edition crate and a new one behave differently on identical source. If you
copy a let-chain into an edition-2021 project, it will not compile.


In [ ]:
// 12.3 Let-chains (edition 2024): `&&` between patterns and conditions.

// Without let-chains this is a nested `if`: the outer one to unwrap, the inner
// one to test a condition that only makes sense once it is unwrapped.
let cfg: Option<(&str, i32)> = Some(("debug", 3));
if let Some((name, level)) = cfg && level > 2 && name.len() > 3 {
    println!("let chain matched: {name} level {level}");
}

// The chain short-circuits, so `level` in the second condition is guaranteed to
// be initialised -- and this version simply does not match:
let quiet: Option<(&str, i32)> = Some(("quiet", 1));
if let Some((name, level)) = quiet && level > 2 {
    println!("unreachable: {name} {level}");
} else {
    println!("the condition after `&&` was false, so we landed in `else`");
}

// It works with `while let` too, which is how you drain a source while some
// separate state still permits it.
let mut queue = vec![5, 4, 3, 2, 1];
let mut allowance = 3;
let mut processed = Vec::new();
while let Some(next) = queue.pop() && allowance > 0 {
    allowance -= 1;
    processed.push(next);
}
println!("processed = {processed:?}, queue left = {queue:?}");

// Chains compose with ordinary conditions on either side, and with `let ...
// else` for the bail-out form:
let pair = Some((2, 7));
if let Some((a, b)) = pair && a < b {
    println!("ascending: {a} < {b}");
}

let earlier = 1 < 2;
if earlier && let Some((a, b)) = pair {
    println!("chained after a plain bool too: {a}, {b}");
}


## 13. Blocks as Expressions, and Diverging Expressions

Every `{ ... }` is an expression. Its value is its **last expression**, provided that
expression is not followed by a semicolon. Drop a `;` on the end and the block's value
becomes `()`, discarding whatever the expression produced. This single rule explains
a great deal of otherwise confusing Rust:

``` rust
let a = { 1 + 2 };      // a is 3
let b = { 1 + 2; };     // b is ()  -- the semicolon threw the value away
```

It is the same rule that makes a function body work without a `return`: the tail
expression is the return value. And it is why a `match` or `if` used as an expression
must have a place to put its result.

Any expression can end a block this way, so `if`/`else`, `match`, `loop` with a
`break` value, and labelled blocks are all just variations on the same idea. Blocks
are also *scopes*: bindings introduced inside are dropped at the closing brace, which
makes a block the natural way to confine a temporary.

### Diverging expressions and the never type

Some expressions never produce a value at all. Their type is written `!` and read
**never**: the program either does not return, or leaves through a different path.

| Expression | Behaviour |
|---|---|
| `return v` | leaves the enclosing function |
| `break` / `continue` | leaves the loop iteration |
| `panic!("…")` | aborts the thread with a message |
| `unreachable!()` | panics, asserting this cannot happen |
| `todo!()` | panics with "not yet implemented" |
| `unimplemented!()` | panics, for a placeholder you owe |

`!` is a subtype of every other type, so a diverging expression can appear wherever any
type was expected. That is what lets `match` arms mix `todo!()` with real values, and
what lets `loop { }` stand in for a function's return value when the loop is the only
thing the function ever does. It is also what makes `let ... else` legal: its `else`
block has type `!`, so it satisfies any requirement the surrounding context has.


In [ ]:
// 13.1 Blocks produce values; the semicolon discards them.

// The tail expression is the value -- unless a `;` follows it.
let with_tail = { 1 + 2 };
let with_semicolon = { println!("  (a side effect, and then a semicolon)"); };
println!("with_tail = {with_tail:?}, with_semicolon = {with_semicolon:?}");

// Discarding a value with `;` is not merely untidy -- rustc complains when the
// value looks like it took work to produce:
//
//     let discarded = { 1 + 2; };
//     warning: unused arithmetic operation that must be used
//      --> probe.rs:2:28
//       |
//     2 |     let discarded = { 1 + 2; };
//       |                        ^^^^^ the arithmetic operation produces a value
//       |
//       = note: `#[warn(unused_must_use)]` (part of `#[warn(unused)]`) on by default
//     help: use `let _ = ...` to ignore the resulting value
//       |
//     2 |     let discarded = { let _ = 1 + 2; };
//       |                        +++++++
//
// `let _ = ...` is the explicit way to say "I know, throw it away".
let discarded = { let _ = 1 + 2; };
println!("discarded = {discarded:?}");

// Blocks are scopes: `scratch` dies at the closing brace, and the value
// escapes without the binding escaping.
let escaped = {
    let scratch = 6 * 7;
    scratch + 100
};
println!("escaped = {escaped}");
// `scratch` is not in scope here -- uncomment to see E0425.

// A block whose last expression is an `if` is just a value:
let kind = {
    let n = 12;
    if n % 3 == 0 { "divisible by 3" } else { "not divisible by 3" }
};
println!("kind = {kind}");

// `return` leaves the enclosing FUNCTION, not the block. This is the usual way
// to write a guard clause before the main body of a function.
fn sign_label(n: i32) -> &'static str {
    if n < 0 {
        return "negative";
    }
    if n == 0 {
        return "zero";
    }
    "positive"
}
for n in [-4, 0, 9] {
    println!("  sign_label({n}) = {}", sign_label(n));
}

// Diverging expressions have type `!` and can stand in for any type, which is
// why `todo!()` compiles inside a `match` arm of type `i32`:
fn mandatory(flag: bool) -> i32 {
    match flag {
        true => 1,
        false => unreachable!(),     // type `!`, coerces to `i32`
    }
}
println!("mandatory(true) = {}", mandatory(true));
// `mandatory(false)` would panic -- that is the point of `unreachable!()`.

fn mandatory_todo(flag: bool) -> i32 {
    match flag {
        true => 2,
        false => todo!("handle the false case"),   // compiles, panics if reached
    }
}
println!("mandatory_todo(true) = {}", mandatory_todo(true));

// `loop` that never breaks has type `!` too, so a function that only ever
// loops needs no return statement:
fn spin_at_most(limit: u32) -> ! {
    let mut n = 0;
    loop {
        n += 1;
        if n >= limit {
            panic!("spun {n} times and gave up");
        }
    }
}
// Not called: it panics by design. Taking it as a function value is enough to
// show that the signature `-> !` is honest, and to keep it from being dead code.
let declared_never: fn(u32) -> ! = spin_at_most;
println!("declared_never is a fn pointer: {} bytes", std::mem::size_of_val(&declared_never));

// A block used purely to confine a temporary, discarding the value:
let mut log = Vec::new();
{
    let temporary = String::from("close me");
    log.push(temporary.len());
}   // `temporary` released here
println!("log = {log:?}");


## 14. Control Flow over `Option` and `Result`

Everything in this chapter was groundwork for two types you have already met and
probably not yet used properly.

- **`Option<T>`** is `Some(T)` or `None`. There is no null in Rust, so "a value that
  might be absent" is spelled out in the type.
- **`Result<T, E>`** is `Ok(T)` or `Err(E)`. There are no exceptions, so "an operation
  that might fail" is spelled out in the type too.

Both are ordinary enums — the enum machinery is chapter 10 — which means every tool in
this chapter applies to them directly. `match`, `if let`, `while let`, `let ... else`
and `matches!` all work on `Option` and `Result` with no special support, and that is
the entire design: error handling is built from the control flow you already have
rather than from a language feature bolted alongside it.

### The three ways to take one apart

| Approach | Right when |
|---|---|
| `match` | you handle every case and each case has real work to do |
| `if let` / `let ... else` | one case is interesting and the other is "ignore" or "bail" |
| combinators (`.map`, `.unwrap_or`, …) | you want to transform the value and keep the wrapper |

There is one more, which is the reason `Result` composes so well: **`?`**. Inside a
function returning `Option` or `Result`, `expr?` either yields the payload or
immediately returns early with the `None`/`Err`. It is `let ... else { return … }`
with the boilerplate removed, and it is how Rust codebases avoid deep nesting without
resorting to unchecked unwrapping.

### `unwrap` is a promise, not a feature

`unwrap()` and `expect()` extract the payload and **panic** if there is none.
`expect("message")` prints your message; `unwrap()` prints a generic one. They are
legitimate when the absence is genuinely impossible or genuinely a bug — and they are
the single most common mistake made by programmers arriving from languages with
exceptions, where the failure would at least have been throwable and catchable. The
idiomatic alternatives are `unwrap_or(default)`, `unwrap_or_else(|| …)`,
`unwrap_or_default()`, and `?`.



In [ ]:
// 14.1 Control flow over `Option` and `Result`.

// --- `match`: every case, with real work in each ---
fn describe_option(maybe: Option<i32>) -> String {
    match maybe {
        Some(n) if n > 0 => format!("positive {n}"),
        Some(n) => format!("non-positive {n}"),
        None => "nothing there".to_string(),
    }
}
println!("{}", describe_option(Some(5)));
println!("{}", describe_option(Some(-5)));
println!("{}", describe_option(None));

// --- `if let` / `let ... else`: one interesting case ---
let maybe: Option<&str> = Some("configured");
if let Some(text) = maybe {
    println!("if let saw: {text}");
}
let text = maybe.unwrap_or("default");
println!("unwrap_or gave: {text}");

// --- combinators: transform and keep the wrapper ---
let doubled = Some(21).map(|n| n * 2);
println!("Some(21).map(x2)  = {doubled:?}");
println!("None.map(x2)       = {:?}", None::<i32>.map(|n| n * 2));
println!("Some(4).filter(>2) = {:?}", Some(4i32).filter(|n| *n > 2));
println!("Some(\"12\").and_then(parse) = {:?}", Some("12").and_then(|s| s.parse::<i32>().ok()));

// --- `?`: the early return, with the noise removed ---
// Compare this:
fn sum_texts_explicit(parts: &[&str]) -> Option<i32> {
    let mut total = 0;
    for part in parts {
        let value = match part.trim().parse::<i32>() {
            Ok(v) => v,
            Err(_) => return None,          // manual early return
        };
        total += value;
    }
    Some(total)
}
// ... with this. Same behaviour, one line of real logic per step:
fn sum_texts(parts: &[&str]) -> Option<i32> {
    let mut total = 0;
    for part in parts {
        total += part.trim().parse::<i32>().ok()?;   // `?` returns None on failure
    }
    Some(total)
}
println!("{:?} / {:?}", sum_texts_explicit(&["1", "2", "3"]), sum_texts(&["1", "2", "3"]));
println!("{:?} / {:?}", sum_texts_explicit(&["1", "x"]), sum_texts(&["1", "x"]));

// `?` works on `Result` too, and `Err` propagates the error value itself --
// which is how error types reach the caller intact:
fn parse_all(parts: &[&str]) -> Result<Vec<i32>, std::num::ParseIntError> {
    let mut out = Vec::new();
    for part in parts {
        out.push(part.trim().parse::<i32>()?);
    }
    Ok(out)
}
println!("{:?}", parse_all(&["4", "5"]));
println!("{:?}", parse_all(&["4", "oops"]));

// `ok_or` / `ok_or_else` convert `Option` into `Result` when you want `?` to
// carry a reason rather than a bare absence:
fn must_be_positive(n: i32) -> Result<i32, String> {
    let checked = if n > 0 { Some(n) } else { None };
    checked.ok_or_else(|| format!("{n} is not positive"))
}
println!("{:?}", must_be_positive(3));
println!("{:?}", must_be_positive(-3));

// `unwrap` is a promise. Both of these are fine because the guarantee is real:
println!("Some(9).unwrap() = {}", Some(9).unwrap_or(0));
println!("Ok(9).unwrap()   = {}", Ok::<i32, &str>(9).unwrap_or(0));
// These would panic -- correctly, since "this cannot be None" was wrong:
//     None::<i32>.unwrap();
//     None::<i32>.expect("the config must have a port");


## 15. Reading the Compiler on Control Flow

This chapter has deliberately quoted rustc verbatim six times, because the messages
are better documentation than any summary. Here they are in one place, with what each
one is actually telling you.

| Code | Message | What it means | Fix |
|---|---|---|---|
| `E0308` | `expected `bool`, found integer` | the condition is not a `bool` — no truthiness (§3) | write the comparison |
| `E0308` | `expected `bool`, found `()`` + *did you mean `==`?* | assignment where a comparison was intended | `=` → `==` |
| `E0308` | ``if` and `else` have incompatible types` | the arms disagree, so the `if` has no single type (§1) | make both arms agree |
| `E0317` | ``if` may be missing an `else` clause` | an `if` without `else` is `()`, and cannot be bound (§1) | add the `else` |
| `E0004` | `non-exhaustive patterns` | a `match` misses cases; the uncovered set is listed (§9) | add the arm, or `_` |
| `E0268` | ``break` outside of a loop or labeled block` | `break` with nothing to break out of (§4) | move it inside, or label a block |
| `E0308` | ``else` clause of `let...else` does not diverge` | the `else` block produces a value instead of leaving (§12) | `return`, `continue` or `panic!` |
| `warn` | `unnecessary parentheses around `if` condition` | C-style parens — not an error, but not the house style | delete them |
| `warn` | `unreachable pattern` | an earlier arm already covers this one (§10) | delete the dead arm |

### How to actually read one

Four habits, in the order that saves the most time:

1. **Fix the first error only, then recompile.** Compiler errors cascade, and later
   errors are frequently consequences of the first. Fixing them one at a time is
   usually faster than fixing them all at once.
2. **Read the caret.** The `^^^^` marks the exact span. It is often shorter than you
   expected, which is itself information — `found `()`` pointing at your `if` says the
   *whole expression* has the wrong type, not the body.
3. **Read the `help:` block.** It is machine-generated and frequently a paste-ready
   patch. `E0308` on `if y = 5` gives you the `+` insertion; `E0004` gives you the arm.
4. **When two errors seem contradictory, suspect the first one's fix.** `E0308` in a
   `match` arm and an `E0004` on the same `match` are usually one problem, not two.

One honest caveat about this notebook: evcxr compiles each cell in isolation, so an
error you make in cell 30 does not stop cell 31 from running. In a real crate the same
mistake stops the whole build. The diagnostics are identical; only the blast radius
differs.


In [ ]:
// 15.1 A worked repair, and the warning everybody hits first.

// The warning you will see within your first hour, because it costs nothing to
// write parens and they are compulsory in C, C++ and Java:
//
//     warning: unnecessary parentheses around `if` condition
//      --> probe.rs:3:8
//       |
//     3 |     if (t > 18) {
//       |        ^      ^
//       |
//       = note: `#[warn(unused_parens)]` (part of `#[warn(unused)]`) on by default
//     help: remove these parentheses
//       |
//     3 -     if (t > 18) {
//     3 +     if t > 18 {
//       |
//
// Identical for `while (x < 3)`. Note the two carets: rustc marks the opening
// and closing parenthesis rather than the expression, and the help block shows
// a unified diff you can apply by hand.
let t = 21;
if t > 18 {
    println!("warm");
}

// --- a worked repair: read the caret before the words ---
//
// Suppose you write a function that grades a score, with no final `else`:
//
//     fn bad(score: u32) -> &'static str {
//         let grade = if score >= 90 { 'A' }
//                     else if score >= 80 { 'B' };
//         grade
//     }
//
// The error, in full:
//
//     error[E0317]: `if` may be missing an `else` clause
//      --> probe.rs:3:22
//       |
//     3 |                 else if score >= 80 { 'B' };
//       |                      ^^^^^^^^^^^^^^^^^---^^
//       |                      |                |
//       |                      |                found here
//       |                      expected `char`, found `()`
//       |
//       = note: `if` expressions without `else` evaluate to `()`
//       = help: consider adding an `else` block that evaluates to the expected type
//
// Note where the caret landed: on the **inner** `if`, not the outer one. rustc
// is pointing at the branch that has no `else`, which is the innermost one --
// the outer `if` does have an `else`, it is just an `else if`. Read the caret
// before the words and the message stops being surprising.
//
// The `note:` states the rule (`if` without `else` is `()`), and the `help:`
// is the fix. Adding the missing `else` resolves it entirely:
fn grade_of(score: u32) -> &'static str {
    if score >= 90 {
        "A"
    } else if score >= 80 {
        "B"
    } else {
        "C"
    }
}
for score in [95u32, 85, 50] {
    println!("  {score} -> {}", grade_of(score));
}

// Note what did *not* happen: no error about `grade` being `char` while the
// function promises `&str`. The type checker stopped at the first error rather
// than cascading, which is the behaviour the fourth habit below relies on.
//
// The general lesson: rustc's first error is usually the real one, its `note:`
// explains the language rule you violated, and its `help:` is often the patch.
// When two errors look unrelated, fix the first and recompile before reading
// the rest.


## 16. Cheat Sheet

### Decisions

``` rust
if cond { }                     // no parens; braces mandatory; yields ()
if cond { a } else { b }        // no ternary — this IS the ternary
if c1 { a } else if c2 { b } else { d }   // first true branch wins

let x = if cond { 1 } else { 2 };          // usable as an expression
let x = if cond { 1 };                     // E0317: no single type
let x = if cond { 1 } else { "a" };        // E0308: arms must agree
```

### Loops

``` rust
loop { }                        // unconditional; only `break`/`return`/panic exits
let v = loop { break 7; };      // `loop` is the ONLY loop that breaks with a value
while cond { }                  // condition must be bool
while let Some(x) = it.next() { }

for item in collection { }      // consumes the collection
for item in &collection { }     // shared borrow
for item in &mut collection { *item += 1; }   // mutable borrow, note the `*`
for i in 0..n { }               // 0 .. n-1      (half-open)
for i in 0..=n { }              // 0 .. n        (inclusive)
for (i, x) in xs.iter().enumerate() { }
for (a, b) in xs.iter().zip(ys) { }
for _ in 0..3 { }               // "three times"

'outer: for x in xs {
    for y in ys {
        break 'outer;           // leaves both loops
        continue 'outer;        // next x
    }
}
```

### `match`

``` rust
match value {
    0            => "zero",
    1 | 2        => "one or two",
    3..=9        => "small",
    'a'..='z'    => "lowercase",       // char ranges
    (0, 0)       => "origin",          // tuple patterns
    [x]          => "one-element slice",
    [first, rest @ ..] => "first plus tail",
    n @ 3..=9    => "names the value",
    x if x > 100 => "guarded",         // guard: falls through to the next arm
    _            => "everything else", // required for i32, optional for bool
}
```

### `if let`, `let ... else`, `matches!`

``` rust
if let Some(x) = opt { }              // handle one case, ignore the rest
if let Ok(v) = res { } else { }       // optional else
let Some(x) = opt else { return; };   // else MUST diverge
let Ok(v) = s.parse() else { continue };

matches!(x, 1..=9)                    // -> bool
matches!(opt, Some(n) if n > 0)
matches!(status, Some(200..=299))

if let Some((name, level)) = cfg && level > 2 { }   // let-chain, edition 2024
```

### `Option` / `Result` control flow

``` rust
match opt { Some(v) => …, None => … }
if let Some(v) = opt { … }
opt.map(|v| v * 2)                    // Option<U>; None stays None
opt.and_then(|v| f(v))                // flatten a nested Option
opt.filter(|v| *v > 0)
opt.unwrap_or(0) / opt.unwrap_or_else(|| compute()) / opt.unwrap_or_default()
opt.ok_or_else(|| "reason".to_string())         // Option -> Result
opt.unwrap() / opt.expect("msg")      // PANICS if None — a promise, not a feature
fn f() -> Option<T> { let v = g()?; Some(v) }   // `?` early-returns None
```

### Discarding and diverging

``` rust
{ expr }        // block value
{ expr; }       // value discarded -> ()  (warns if the value took work)
let _ = expr;   // explicit discard
return v;  break;  continue;            // type `!`
panic!("msg");  unreachable!();  todo!();  unimplemented!();
loop { }                                // type `!` when it never breaks
```

### The diagnostic codes worth remembering

`E0308` mismatched types (covers half of the mistakes in this chapter) · `E0317` `if` without `else` in a value position · `E0004` non-exhaustive `match` · `E0268` `break` outside a loop · `E0382` use after move · `warn(unused_parens)` on `if (x)` · `warn(unreachable_patterns)`.


In [ ]:
// 16.1 Every construct in this chapter, once, on one input.

let input = 37;

// 1. `if` as an expression, with parens omitted and arms agreeing.
let size = if input > 50 { "large" } else if input > 10 { "medium" } else { "small" };

// 2. A `loop` that breaks with a value.
let mut probe = 1u32;
let first_square_over = loop {
    if probe * probe > input as u32 {
        break probe * probe;
    }
    probe += 1;
};

// 3. Labelled nested loops with `continue` on the outer one.
let mut products = Vec::new();
'tab: for a in 1..=3 {
    for b in 1..=3 {
        if a * b % 2 != 0 {
            continue 'tab;
        }
        products.push(a * b);
    }
}

// 4. `while let` draining a source.
let mut digits = Vec::new();
let mut remaining = input;
while let Some(d) = if remaining > 0 { Some(remaining % 10) } else { None } {
    remaining /= 10;
    digits.push(d);
}

// 5. `match` with a range, a guard, an or-pattern and `@`.
let band = match input {
    n @ 0..=9 => format!("{n}: units"),
    n @ 10..=99 if n % 2 == 0 => format!("{n}: two digits, even"),
    n @ 10..=99 => format!("{n}: two digits, odd"),
    n => format!("{n}: large"),
};

// 6. `if let` on a tuple pattern, `let ... else` with a bail-out, `matches!`.
let maybe_pair = Some((3, 9));
let mut note = String::from("no pair");
if let Some((x, y)) = maybe_pair {
    note = format!("pair ({x}, {y})");
}
fn classify_parse(text: &str) -> &'static str {
    let Ok(v) = text.parse::<i32>() else {
        return "not an integer";
    };
    if matches!(v, 1..=99) { "in range" } else { "out of range" }
}

// 7. `Option`/`Result` control flow with `?`.
fn sum_parsed(parts: &[&str]) -> Option<i32> {
    let mut total = 0;
    for p in parts {
        total += p.parse::<i32>().ok()?;
    }
    Some(total)
}

println!("size                = {size}");
println!("first_square_over   = {first_square_over}");
println!("products            = {products:?}");
println!("digits (reversed)   = {digits:?}");
println!("band                = {band}");
println!("note                = {note}");
println!("classify_parse(\"7\") = {}", classify_parse("7"));
println!("classify_parse(\"z\") = {}", classify_parse("z"));
println!("sum_parsed          = {:?}", sum_parsed(&["1", "2", "3"]));
println!("sum_parsed (bad)    = {:?}", sum_parsed(&["1", "x"]));


### Coming from Python or C++

| You would write | In Rust | Note |
|---|---|---|
| `if x:` | `if x != 0` / `if !s.is_empty()` | no truthiness (§3) |
| `if not items:` | `if items.is_empty()` | §3 |
| `x = 5 if c else 9` / `c ? 5 : 9` | `if c { 5 } else { 9 }` | no `?:` (§1) |
| `if (x = 5):` | `if x == 5` | `=` is `E0308`, not a warning (§3) |
| `if x: … elif y: …` / `else if` | `else if` — the same words | it's a nested `if`, not a keyword (§2) |
| `switch` with `break` | `match` — no fallthrough | §9 |
| `default:` | `_ =>` | required unless exhaustive (§9) |
| `case 1 ... 5:` (GNU C) | `1..=5 =>` | a real pattern, checked for overlap (§10) |
| `while (1)` / `for(;;)` | `loop { }` | §4 |
| `do { } while (c);` | `loop { … if !c { break; } }` | §6 |
| `for (int i = 0; i < n; ++i)` | `for i in 0..n` | §7 |
| `for (i, x) in enumerate(xs)` | `for (i, x) in xs.iter().enumerate()` | §7 |
| `for k, v in d.items():` | `for (k, v) in &map` | chapter 06 |
| `break 2` (Perl) / `goto` | `break 'outer` | labelled loops (§5) |
| `continue 2` | `continue 'outer` | §5 |
| `out = None` then `if out is not None` | `let out = loop { … break v; };` | no sentinel needed (§4) |
| `x = f(); if x is None: return` | `let x = f() else { return };` | diverging `else` (§12) |
| `if x is not None:` | `if let Some(x) = x_opt` | §12 |
| `isinstance(x, T)` | `match x { T(..) => … }` | chapter 10 |
| `x if cond else y` for validation | `let x = v.ok_or_else(|| …)?` | §14 |
| `try: … except: …` | `match` / `?` / `.unwrap_or_else` | no exceptions (§14) |
| `x?.y` (optional chaining) | `x.map(\|v\| v.y)` or `x?` in a returning fn | §14 |
| `assert` / `throw` in unreachable code | `unreachable!()`, `todo!()` | type `!`, coerces anywhere (§13) |



### What Comes Next

You now have the whole of Rust's control flow: expressions that produce values,
conditions that must be booleans, three loops with different jobs, a pattern language
with a completeness check, and the short forms (`if let`, `let ... else`, `matches!`,
`?`) that keep all of it from becoming nested noise.

Two loose ends from chapter 01 are now tied off. The `if`-as-expression idea promised
in `01`'s §14 has had its full treatment — including why the arms must agree and why
an `if` without an `else` is `()` — and `Option` and `Result` from `01`'s §7.4 finally
have an honest way to be taken apart, in [§14](#14-control-flow-over-option-and-result).

- **Chapter 03 — Functions.** Parameters, return types, and the tail-expression rule
  (§13 here) as *the* way functions return values. Multiple returns via tuples, `const
  fn`, and the first look at closures — which are values, and therefore need everything
  in this chapter to be usable.
- **Chapter 04 — Strings.** `String` versus `&str`, UTF-8, and why `.len()` counts
  bytes while `.chars()` counts characters (you saw the first two bytes of `"héllo"` in
  §7; chapter 04 explains them).
- **Chapter 05 — Ownership and borrowing.** The rule behind §8: why `for x in vec`
  consumes it, why `&mut` needs the `*`, and why `E0382` — the one error in this
  chapter you were shown rather than made to fix — is the error Rust is famous for.

Then chapter 06 collects the native collections, chapter 07 covers I/O, chapter 08
covers assertions and type checks, and chapter 10 closes the loop with structs, enums
and traits — at which point `Option` and `Result` stop being magical and become
ordinary enums you can write yourself.

### Before you move on

The practice notebook is `02_logic-blocks-and-control-flow_p.ipynb`. It follows the
same section order and asks you to write the code rather than read it — 21 exercises,
each with a stub, a doc comment, and a block of checks that prints `Exercise N.M passed`
when your implementation is right.

Three suggestions before you open it:

- **Re-run this notebook from a fresh kernel first.** Restart, run all, and watch for
  anything that behaves differently from the first pass. A notebook is only as
  trustworthy as its last clean top-to-bottom run.
- **Write the exercises without looking back here first.** The lookups you need are
  few, and the ones you have to look up are the ones you had not internalised.
- **Make the failing checks fail loudly.** Nothing in the practice notebooks is hidden
  from you: the expected values are literal, so if a check passes for the wrong reason
  you will be able to see it.

